In [1]:
import os
import sys

if "google.colab" in sys.modules:
    # Running in Colab

    !git clone https://github.com/pthengtr/kcw-analytics.git
    !cd /content/kcw-analytics && git pull origin main

    from google.colab import drive
    drive.mount("/content/drive")

    BASE_FOLDER = "/content/drive/Shareddrives"
    BASE_FOLDER_GIT = "/content"
else:
    # Running in local Jupyter
    BASE_FOLDER = r"G:\Shared drives"
    BASE_FOLDER_GIT = r"C:\Users\Windows 11\Notebook"

print("Using folder:", BASE_FOLDER)

Using folder: G:\Shared drives


In [2]:

folder = f"{BASE_FOLDER}/KCW-Data/kcw_analytics/01_raw"

In [3]:
import os
import pandas as pd

data = {}

for file in os.listdir(folder):
    if file.endswith(".csv"):
        path = os.path.join(folder, file)
        data[file] = pd.read_csv(
            path,
            dtype={
              "BCODE": "string",
              "ITEMNO": "string",
              "BILLNO": "string",
            },
            encoding="utf-8-sig",
            low_memory=False   # stops chunk guessing
        )
        print(f"Loaded: {file} -> {data[file].shape}")



Loaded: raw_inventory_hq_2024.csv -> (4983, 8)
Loaded: raw_syp_simas_sales_bills.csv -> (20146, 49)
Loaded: raw_hq_pidet_purchase_lines.csv -> (233088, 41)
Loaded: raw_syp_sidet_sales_lines.csv -> (54110, 38)
Loaded: raw_syp_pimas_purchase_bills.csv -> (3818, 49)
Loaded: raw_hq_simas_sales_bills.csv -> (277372, 49)
Loaded: raw_hq_pimas_purchase_bills.csv -> (74572, 49)
Loaded: raw_hq_sidet_sales_lines.csv -> (733556, 38)
Loaded: raw_syp_pidet_purchase_lines.csv -> (35177, 41)
Loaded: raw_hq_icmas_products.csv -> (115635, 94)
Loaded: raw_hq_pvmas_notes_vouchers.csv -> (14456, 32)
Loaded: raw_hq_armas_receivable.csv -> (2321, 20)
Loaded: raw_hq_apmas_payable.csv -> (988, 20)
Loaded: raw_hq_rvmas_notes_vouchers.csv -> (14456, 32)
Loaded: raw_syp_icmas_products.csv -> (38724, 94)


In [4]:

import sys
import importlib

# ensure repo is on path
repo_path = f"{BASE_FOLDER_GIT}/kcw-analytics"
if repo_path not in sys.path:
    sys.path.append(repo_path)

# import the module (NOT individual functions)
import src.kcw.utils as utils

# reload to pick up latest .py changes
importlib.reload(utils)



<module 'src.kcw.utils' from 'C:\\Users\\Windows 11\\Notebook/kcw-analytics\\src\\kcw\\utils.py'>

In [5]:
df_simas = data["raw_hq_simas_sales_bills.csv"].copy()
df_pimas = data["raw_hq_pimas_purchase_bills.csv"].copy()

In [6]:
df_simas.columns

Index(['ID', 'JOURMODE', 'JOURTYPE', 'JOURDATE', 'JOURNO', 'JOURTIME',
       'DEPTNO', 'BOOKNO', 'BILLTYPE', 'BILLDATE', 'BILLTIME', 'BILLNO',
       'LINES', 'TAXIC', 'DISCOUNT', 'DEDUCT', 'BEFORETAX', 'VAT', 'TAX',
       'AFTERTAX', 'EXEMPT', 'SVCCHG', 'WITHHOLD', 'PAID', 'CASHED', 'CASHAMT',
       'CHKAMT', 'DUEAMT', 'PAYSTAT', 'ACCTNO', 'ACCTNAME', 'ADDR1', 'ADDR2',
       'PO', 'SALE', 'RE', 'TERM', 'DUEDATE', 'NOTEDATE', 'NOTENO',
       'VOUCDATE1', 'VOUCNO1', 'VOUCDATE2', 'VOUCNO2', 'POSTED1', 'POSTED2',
       'REMARKS', 'CANCELED', 'DONE'],
      dtype='object')

In [7]:
# Normalize to midnight so BILLDATE comparisons don't include the 1st of the cutoff month
# (e.g. April 1 00:00 must be excluded from the as-of-March-31 report).
cutoff = pd.Timestamp.today().normalize().replace(day=1)

#cutoff = pd.Timestamp(2026, 2, 1)

YEAR = cutoff.year
MONTH = cutoff.month

cutoff

Timestamp('2026-07-01 00:00:00')

In [8]:
def normalize_acctname(name):
    if pd.isna(name):
        return name

    n = str(name).lower()

    for k in ["lazada", "shopee", "tiktok"]:
        if k in n:
            return f"คุณลูกค้าทั่วไป {k}"

    return name


In [9]:
df = df_simas

df["BILLDATE"] = pd.to_datetime(df["BILLDATE"].astype(str).str.strip(), errors="coerce")
df["VOUCDATE2"] = pd.to_datetime(df["VOUCDATE2"], errors="coerce")

mask_billno = df["BILLNO"].str.contains(r"TD|TR|TAD|CN", na=False)
mask_due_pos = df["DUEAMT"] != 0
mask_billbefore = df["BILLDATE"] < cutoff
mask_paidafter = df["VOUCDATE2"] >= cutoff
mask_null = df["VOUCDATE2"].isna()
mask_cancel = df["CANCELED"].astype(str).str.strip().str.upper() == "N"
mask_vat = pd.to_numeric(df["VAT"], errors="coerce").fillna(0) > 0

df["ACCTNAME"] = df["ACCTNAME"].apply(normalize_acctname)

df_ar_sale = df[mask_billno & mask_due_pos & mask_billbefore & (mask_null | mask_paidafter) & mask_cancel & mask_vat]

df_ar_sale = df_ar_sale.sort_values(by=["ACCTNAME", "BILLDATE"])

In [10]:
df_ar_sale[["BILLNO", "BILLDATE","ACCTNO", "ACCTNAME", "VOUCDATE1", "VOUCDATE2", "PAID","DUEAMT", "VAT"]]

,BILLNO,BILLDATE,ACCTNO,ACCTNAME,VOUCDATE1,VOUCDATE2,PAID,DUEAMT,VAT
249506,TAD6902-059,2026-02-03,7LPNT,คุณลูกค้าทั่วไป lazada,NaN,NaT,N,3990.00,7.0
250824,TAD6902-251,2026-02-11,7LAZ1,คุณลูกค้าทั่วไป lazada,NaN,NaT,N,5040.00,7.0
251111,TAD6902-281,2026-02-12,7LPNT,คุณลูกค้าทั่วไป lazada,NaN,NaT,N,1850.00,7.0
251174,TAD6902-289,2026-02-13,7LPNT,คุณลูกค้าทั่วไป lazada,NaN,NaT,N,1750.00,7.0
251681,TAD6902-362,2026-02-16,7LAZ1,คุณลูกค้าทั่วไป lazada,NaN,NaT,N,375.00,7.0
...,...,...,...,...,...,...,...,...,...
273457,TD6906-129,2026-06-26,7กช,หจก. ส.กิจชัยคอนกรีต (สาขาที่ 00002),NaN,2026-07-18,Y,250.00,7.0
273935,TD6906-144,2026-06-29,7กช,หจก. ส.กิจชัยคอนกรีต (สาขาที่ 00002),NaN,2026-07-18,Y,3250.00,7.0
261223,TD6904-053,2026-04-17,7จยน,ห้างหุ้นส่วนจำกัด จี.ยูเนี่ยน (สำนักงานใหญ่),NaN,2026-07-08,Y,32940.00,7.0
268962,TD6906-005,2026-06-01,7จยน,ห้างหุ้นส่วนจำกัด จี.ยูเนี่ยน (สำนักงานใหญ่),NaN,NaT,N,27461.66,7.0


In [11]:
df = df_ar_sale

df_ar_summary = (
    df
    .groupby("ACCTNAME")
    .agg(
        bills=("BILLNO", "nunique"),
        total_amount=("DUEAMT", "sum")
    )
    .reset_index()
)

In [12]:
df_ar_summary

,ACCTNAME,bills,total_amount
0,คุณลูกค้าทั่วไป lazada,560,631925.00
1,คุณลูกค้าทั่วไป shopee,259,263336.00
2,คุณลูกค้าทั่วไป tiktok,326,156049.90
3,บริษัท 168 เทรลเลอร์ทรานสปอร์ต จำกัด (สำนักงาน...,7,38670.50
4,บริษัท ย่งฮวดกรุ๊ป 7895 จำกัด (สำนักงานใหญ่),2,16295.00
5,บริษัท ว.วิมา ทิมเบอร์ จำกัด (สำนักงานใหญ่),4,7811.01
6,บริษัท ส.พลายวู๊ด จำกัด (สำนักงานใหญ่),1,1800.00
7,บริษัท สหกิจ แกลง จำกัด (สาขาที่ 00001),3,11641.90
8,บริษัท สหกิจ แกลง จำกัด (สำนักงานใหญ่),4,33240.61
9,บริษัท สหกิจตราด จำกัด (สำนักงานใหญ่),5,22241.30


In [13]:
df = df_pimas

df["BILLDATE"] = pd.to_datetime(df["BILLDATE"].astype(str).str.strip(), errors="coerce")
df["VOUCDATE2"] = pd.to_datetime(df["VOUCDATE2"], errors="coerce")

mask_due_pos = df["DUEAMT"] != 0
mask_billbefore = df["BILLDATE"] < cutoff
mask_paidafter = df["VOUCDATE2"] >= cutoff
mask_null = df["VOUCDATE2"].isna()
mask_cancel = df["CANCELED"].astype(str).str.strip().str.upper() == "N"
mask_vat = pd.to_numeric(df["VAT"], errors="coerce").fillna(0) > 0
# Exclude internal credit-note style AP bills
mask_exclude_billno = ~df["BILLNO"].astype(str).str.upper().str.startswith(("7KCCN", "7KWCN"), na=False)

df["ACCTNAME"] = df["ACCTNAME"].apply(normalize_acctname)

df_ap_purchase = df[mask_due_pos & mask_billbefore & (mask_null | mask_paidafter) & mask_cancel & mask_vat & mask_exclude_billno]

df_ap_purchase = df_ap_purchase.sort_values(by=["ACCTNAME", "BILLDATE"])


In [14]:
df_ap_purchase[["BILLNO", "BILLDATE","ACCTNO", "ACCTNAME", "VOUCDATE1", "VOUCDATE2", "PAID","DUEAMT", "VAT"]]

,BILLNO,BILLDATE,ACCTNO,ACCTNAME,VOUCDATE1,VOUCDATE2,PAID,DUEAMT,VAT
72296,BV 6905000002,2026-05-04,7MSC,บจก. มิตซู สแปร์พาร์ทส เอ็กซ์พอร์ทติ้ง (2012) ...,NaN,NaT,N,5355.35,7.0
72638,BV 6905000008,2026-05-14,7MSC,บจก. มิตซู สแปร์พาร์ทส เอ็กซ์พอร์ทติ้ง (2012) ...,NaN,NaT,N,749.75,7.0
72822,BV 6905000014,2026-05-21,7MSC,บจก. มิตซู สแปร์พาร์ทส เอ็กซ์พอร์ทติ้ง (2012) ...,NaN,NaT,N,7761.14,7.0
73360,BV 6906000009,2026-06-10,7MSC,บจก. มิตซู สแปร์พาร์ทส เอ็กซ์พอร์ทติ้ง (2012) ...,NaN,NaT,N,6529.41,7.0
73485,BV 6906000016,2026-06-15,7MSC,บจก. มิตซู สแปร์พาร์ทส เอ็กซ์พอร์ทติ้ง (2012) ...,NaN,NaT,N,8342.81,7.0
...,...,...,...,...,...,...,...,...,...
73864,IV26060541,2026-06-25,7LK,ห้างหุ้นส่วนจำกัด โลหะกิจกลการ อิมปอร์ต (สำนัก...,NaN,NaT,N,6253.21,7.0
73895,IV26060566,2026-06-26,7LK,ห้างหุ้นส่วนจำกัด โลหะกิจกลการ อิมปอร์ต (สำนัก...,NaN,NaT,N,8186.61,7.0
73774,CN26060007,2026-06-27,7LK,ห้างหุ้นส่วนจำกัด โลหะกิจกลการ อิมปอร์ต (สำนัก...,NaN,NaT,N,-662.46,7.0
73931,IV26060576,2026-06-27,7LK,ห้างหุ้นส่วนจำกัด โลหะกิจกลการ อิมปอร์ต (สำนัก...,NaN,NaT,N,2931.03,7.0


In [15]:
df = df_ap_purchase

df_ap_summary = (
    df
    .groupby("ACCTNAME")
    .agg(
        bills=("BILLNO", "nunique"),
        total_amount=("DUEAMT", "sum")
    )
    .reset_index()
)

In [16]:
df_ap_summary

,ACCTNAME,bills,total_amount
0,บจก. มิตซู สแปร์พาร์ทส เอ็กซ์พอร์ทติ้ง (2012) ...,7,31688.02
1,บจก. เอส ที เค กรุ๊ป 2008 (สำนักงานใหญ่),19,57278.62
2,บจก. เอส.พี.อาร์.วาย ออโต้พาร์ท (สำนักงานใหญ่),28,30960.45
3,บจก.ศรีสยามกลการ (สาขาที่ 00000),53,319047.53
4,บจก.ศรีสยามกลการ (สาขาที่ 00001),1,520.02
...,...,...,...
80,ห้างหุ้นส่วนจำกัด รุ่งเรืองทรัพย์ทวี กลการ (สนญ.),4,16210.50
81,ห้างหุ้นส่วนจำกัด หนองหอยปิโตรเลียม (สำนักงานใ...,1,576.00
82,ห้างหุ้นส่วนจำกัด ฮ.อาหลั่ยยนต์ (สำนักงานใหญ่),7,41564.15
83,ห้างหุ้นส่วนจำกัด เอส.เค.ที. แมชชีนทูลส์ (สนญ.),35,89594.48


In [17]:
def add_total_row(df, label_col="ACCTNAME", amount_col="total_amount"):
    total_row = pd.DataFrame([{
        label_col: "TOTAL",
        "bills": df["bills"].sum(),
        amount_col: df[amount_col].sum()
    }])

    return pd.concat([df, total_row], ignore_index=True)

In [18]:
def rename_and_keep(df, rename_map):
    cols = [c for c in rename_map.keys() if c in df.columns]
    return df[cols].rename(columns=rename_map)

In [19]:
df_ap_summary = add_total_row(df_ap_summary)
df_ar_summary = add_total_row(df_ar_summary)

In [20]:
rename_map = {
    "ACCTNAME": "ชื่อลูกหนี้",
    "BILLNO": "เลขที่บิล",
    "BILLDATE": "วันที่",
    "DUEAMT": "จำนวน",
    "DUEDATE": "ครบกำหนด",
    "bills": "จำนวนบิล",
    "total_amount": "ยอดรวม",
}

df_ar_sale     = rename_and_keep(df_ar_sale, rename_map)
df_ar_summary  = rename_and_keep(df_ar_summary, rename_map)

rename_map["ACCTNAME"] = "ชื่อเจ้าหนี้"

df_ap_purchase      = rename_and_keep(df_ap_purchase , rename_map)
df_ap_summary    = rename_and_keep(df_ap_summary  , rename_map)

In [21]:
TH_MONTHS_ABBR = [
    "ม.ค.", "ก.พ.", "มี.ค.", "เม.ย.", "พ.ค.", "มิ.ย.",
    "ก.ค.", "ส.ค.", "ก.ย.", "ต.ค.", "พ.ย.", "ธ.ค."
]

def thai_date_text(d):
    dt = pd.to_datetime(d)
    return f"{dt.day} {TH_MONTHS_ABBR[dt.month-1]} {dt.year + 543}"

def thai_be_date(d):
    if pd.isna(d):
        return ""

    dt = pd.to_datetime(d)
    return f"{dt.day:02d}/{dt.month:02d}/{dt.year + 543}"

In [22]:
for df in [df_ap_purchase, df_ap_summary, df_ar_sale, df_ar_summary]:

    if "วันที่" in df.columns:
        df["วันที่"] = df["วันที่"].apply(thai_be_date)

    if "ครบกำหนด" in df.columns:
        df["ครบกำหนด"] = df["ครบกำหนด"].apply(thai_be_date)


In [23]:
df_ap_summary["จำนวนบิล"] = df_ap_summary["จำนวนบิล"].astype(int)
df_ar_summary["จำนวนบิล"] = df_ar_summary["จำนวนบิล"].astype(int)

In [24]:
import pandas as pd
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Border, Side, Alignment
from openpyxl.utils import get_column_letter
from openpyxl.worksheet.table import Table, TableStyleInfo


def _export_single_report(
    detail_df,
    summary_df,
    output_file,
    *,
    cutoff,
    thai_date_text,
    report_type,  # "AR" or "AP"
):
    if report_type.upper() == "AR":
        sheets = {
            "AR Summary": summary_df.copy(),
            "AR": detail_df.copy(),
        }
        title_map = {
            "AR Summary": "สรุปลูกหนี้คงค้าง (AR Summary)",
            "AR": "รายงานลูกหนี้คงค้าง (AR Detail)",
        }
    elif report_type.upper() == "AP":
        sheets = {
            "AP Summary": summary_df.copy(),
            "AP": detail_df.copy(),
        }
        title_map = {
            "AP Summary": "สรุปเจ้าหนี้คงค้าง (AP Summary)",
            "AP": "รายงานเจ้าหนี้คงค้าง (AP Detail)",
        }
    else:
        raise ValueError("report_type must be either 'AR' or 'AP'")

    wb = Workbook()
    wb.remove(wb.active)

    # styles
    header_fill = PatternFill("solid", fgColor="1F4E78")
    header_font = Font(name="Tahoma", size=12, bold=True, color="FFFFFF")
    body_font = Font(name="Tahoma", size=11, color="000000")
    total_font = Font(name="Tahoma", size=11, bold=True, color="000000")
    title_font = Font(name="Tahoma", size=14, bold=True)
    subtitle_font = Font(name="Tahoma", size=11)

    thin_gray = Side(style="thin", color="D9D9D9")
    header_border = Border(bottom=Side(style="medium", color="FFFFFF"))
    body_border = Border(bottom=thin_gray)

    fill_group_1 = PatternFill("solid", fgColor="F7FBFF")
    fill_group_2 = PatternFill("solid", fgColor="E8F1FB")
    total_fill = PatternFill("solid", fgColor="FFF2CC")

    header_align = Alignment(horizontal="center", vertical="center")
    text_align = Alignment(horizontal="left", vertical="center")
    num_align = Alignment(horizontal="right", vertical="center")
    date_align = Alignment(horizontal="center", vertical="center")

    amount_headers = {"จำนวน", "ยอดรวม", "DUEAMT", "total_amount", "VAT"}
    count_headers = {"จำนวนบิล", "bills"}
    date_headers = {"วันที่", "ครบกำหนด", "BILLDATE", "VOUCDATE2", "DUEDATE"}

    report_date = thai_date_text(cutoff - pd.Timedelta(days=1))

    title_row = 1
    subtitle_row = 2
    header_row = 4
    data_start_row = header_row + 1

    for sheet_name, df in sheets.items():
        df = df.copy()
        df.columns = df.columns.map(str)

        ws = wb.create_sheet(title=sheet_name)
        title = title_map.get(sheet_name, sheet_name)

        # title
        title_cell = ws.cell(row=title_row, column=1, value=title)
        title_cell.font = title_font
        title_cell.alignment = Alignment(horizontal="center", vertical="center")

        date_cell = ws.cell(row=subtitle_row, column=1, value=f"ณ วันที่ {report_date}")
        date_cell.font = subtitle_font
        date_cell.alignment = Alignment(horizontal="center", vertical="center")

        # merge title/subtitle
        if len(df.columns) > 0:
            ws.merge_cells(
                start_row=title_row,
                start_column=1,
                end_row=title_row,
                end_column=len(df.columns),
            )
            ws.merge_cells(
                start_row=subtitle_row,
                start_column=1,
                end_row=subtitle_row,
                end_column=len(df.columns),
            )

        # write header + data
        rows_to_write = [list(df.columns)] + df.values.tolist()

        for r_idx, row in enumerate(rows_to_write, start=header_row):
            for c_idx, value in enumerate(row, start=1):
                cell = ws.cell(row=r_idx, column=c_idx, value=value)

                if r_idx == header_row:
                    cell.fill = header_fill
                    cell.font = header_font
                    cell.alignment = header_align
                    cell.border = header_border
                else:
                    cell.font = body_font
                    cell.border = body_border

        # alternate row color by name group
        name_cols = ["ชื่อลูกหนี้", "ชื่อเจ้าหนี้", "ACCTNAME"]
        name_col = next((c for c in name_cols if c in df.columns), None)

        if name_col is not None and len(df) > 0:
            current_fill = fill_group_1
            prev_name = None
            name_col_idx = list(df.columns).index(name_col) + 1

            for row_idx in range(data_start_row, data_start_row + len(df)):
                current_name = ws.cell(row=row_idx, column=name_col_idx).value

                if prev_name is None:
                    prev_name = current_name
                elif current_name != prev_name:
                    current_fill = fill_group_2 if current_fill == fill_group_1 else fill_group_1
                    prev_name = current_name

                for col_idx in range(1, len(df.columns) + 1):
                    ws.cell(row=row_idx, column=col_idx).fill = current_fill

        # detect total row
        total_row_idx = None
        if len(df) > 0 and name_col is not None:
            for i, v in enumerate(df[name_col].astype(str), start=data_start_row):
                if v.strip().upper() == "TOTAL":
                    total_row_idx = i
                    break

        # format body cells by column type
        for col_idx, col_name in enumerate(df.columns, start=1):
            for row_idx in range(data_start_row, data_start_row + len(df)):
                cell = ws.cell(row=row_idx, column=col_idx)

                if col_name in date_headers:
                    cell.alignment = date_align

                elif col_name in amount_headers:
                    cell.number_format = '#,##0.00;[Red](#,##0.00);-'
                    cell.alignment = num_align

                elif col_name in count_headers:
                    cell.number_format = '0'
                    cell.alignment = num_align

                elif pd.api.types.is_numeric_dtype(df[col_name]):
                    cell.number_format = '#,##0.00;[Red](#,##0.00);-'
                    cell.alignment = num_align

                else:
                    cell.alignment = text_align

            if total_row_idx is not None:
                ws.cell(total_row_idx, col_idx).font = total_font
                ws.cell(total_row_idx, col_idx).fill = total_fill

        # row heights
        ws.row_dimensions[title_row].height = 24
        ws.row_dimensions[subtitle_row].height = 20
        ws.row_dimensions[header_row].height = 22
        for r in range(data_start_row, data_start_row + len(df)):
            ws.row_dimensions[r].height = 20

        # column widths
        for col_idx, col_name in enumerate(df.columns, start=1):
            col_letter = get_column_letter(col_idx)
            max_len = len(str(col_name))

            for row_idx in range(title_row, data_start_row + len(df)):
                val = ws.cell(row=row_idx, column=col_idx).value
                if val is None:
                    continue

                text = str(val)
                visual_len = len(text) + sum(1 for ch in text if ord(ch) > 127)
                max_len = max(max_len, visual_len)

            if col_name in ["ชื่อลูกหนี้", "ชื่อเจ้าหนี้", "ACCTNAME"]:
                ws.column_dimensions[col_letter].width = min(max_len + 8, 90)
            elif col_name in ["เลขที่บิล", "BILLNO"]:
                ws.column_dimensions[col_letter].width = min(max_len + 4, 24)
            elif col_name in ["วันที่", "ครบกำหนด","BILLDATE", "VOUCDATE2", "DUEDATE"]:
                ws.column_dimensions[col_letter].width = 16
            else:
                ws.column_dimensions[col_letter].width = min(max_len + 3, 22)

        # freeze below header
        ws.freeze_panes = f"A{data_start_row}"

        # add table
        if len(df.columns) > 0 and len(df) > 0:
            end_col = get_column_letter(len(df.columns))
            end_row = header_row + len(df)
            table_ref = f"A{header_row}:{end_col}{end_row}"

            safe_name = sheet_name.replace(" ", "_").replace("-", "_")
            table = Table(displayName=f"Tbl_{safe_name}", ref=table_ref)
            table.tableStyleInfo = TableStyleInfo(
                name="TableStyleMedium2",
                showFirstColumn=False,
                showLastColumn=False,
                showRowStripes=False,
                showColumnStripes=False,
            )
            ws.add_table(table)

    wb.save(output_file)


def export_ar_report(
    df_ar_sale,
    df_ar_summary,
    output_file,
    *,
    cutoff,
    thai_date_text,
):
    _export_single_report(
        detail_df=df_ar_sale,
        summary_df=df_ar_summary,
        output_file=output_file,
        cutoff=cutoff,
        thai_date_text=thai_date_text,
        report_type="AR",
    )


def export_ap_report(
    df_ap_purchase,
    df_ap_summary,
    output_file,
    *,
    cutoff,
    thai_date_text,
):
    _export_single_report(
        detail_df=df_ap_purchase,
        summary_df=df_ap_summary,
        output_file=output_file,
        cutoff=cutoff,
        thai_date_text=thai_date_text,
        report_type="AP",
    )

In [25]:
date_suffix = (cutoff - pd.Timedelta(days=1)).strftime("_%Y_%m_%d")

ar_folder = os.path.join(
    BASE_FOLDER,
    "KCW-Data",
    "kcw_analytics",
    "04_outputs",
    "03_AR_AP_Report",
    f"ar_report{date_suffix}.xlsx"
)

# ⭐ create directory if missing
os.makedirs(os.path.dirname(ar_folder), exist_ok=True)

ap_folder = os.path.join(
    BASE_FOLDER,
    "KCW-Data",
    "kcw_analytics",
    "04_outputs",
    "03_AR_AP_Report",
    f"ap_report{date_suffix}.xlsx"
)

# ⭐ create directory if missing
os.makedirs(os.path.dirname(ar_folder), exist_ok=True)

In [26]:
output_file = folder

export_ar_report(
    df_ar_sale=df_ar_sale,
    df_ar_summary=df_ar_summary,
    output_file=ar_folder,
    cutoff=cutoff,
    thai_date_text=thai_date_text,
)

export_ap_report(
    df_ap_purchase=df_ap_purchase,
    df_ap_summary=df_ap_summary,
    output_file=ap_folder,
    cutoff=cutoff,
    thai_date_text=thai_date_text,
)